# SLR09 - Fine-tune SENA dan evaluasi v0.1

**Workflow:** pulihkan data review -> gunakan baseline tersimpan -> muat SENA v1 dari Drive -> fine-tune 110 train -> pilih checkpoint pada 620 validation -> evaluasi -> ekspor.

**Ini eksperimen baru yang belum dijalankan. Run all akan melatih SENA.** Pilih GPU Colab. NARA tidak dilatih atau diuji ulang. Tidak perlu mengunggah ulang model atau mengisi review lagi. Tiga TUNDA tetap dikecualikan. Validation dipakai memilih checkpoint dan bukan test independen. Tidak mengganti model aktif atau menetapkan threshold produksi.


## 1. Dependensi

Gunakan sesi GPU baru; PyTorch bawaan Colab dipakai. Transformers mengikuti versi yang telah dipakai pada run NARA sebelumnya.


In [ ]:
import os, subprocess, sys
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "accelerate>=1.2,<2", "safetensors>=0.4,<1", "matplotlib", "scikit-learn"])


## 2. Snapshot dan validasi data

Sel ini hanya memulihkan kode/data dan memeriksa input; belum memuat atau melatih model.


In [ ]:
# Restore frozen code/data only; this cell does not run a model.
from pathlib import Path
import base64, hashlib, json, os, sys, zlib
WORK_ROOT = Path(os.environ.get("SLR09_NOTEBOOK_ROOT", '/content/slr09_sena_finetune_v01'))
SNAPSHOT = json.loads(zlib.decompress(base64.b64decode('...')))
EXPECTED = {'code/sena_experiment.py': 'b81db2dccb3584907329690b4e1634f6b309e863ec0e83cd8f1be47c748dde81', 'code/evaluate_sena_reviewed.py': '67d3d443f1e34b9a7881b58fb78cc1c27569398f79927f3f3c8262e3da2fa6cc', 'data/input_manifest.json': '9dd6c6c0ec3c05e477ab1393c2ef180692047bcaf61faadecda4a7d27a92ec0d', 'data/sena_train_reviewed.jsonl': '0e97bdd197442821d5da02cbe4f2ab863781f037e411eb70fddceed1f62d0c18', 'data/sena_validation_reviewed.jsonl': '8055d15c77b0d30ca33f2427e64c6a1d2daf085b6dabd37e8983911fd36e7fc0', 'data/baseline_reviewed_evaluation.json': '20baa5656d59d969d1a76ff8565b8cd8a0c4761df959ba42f38f5458ccfd3643'}
for relative, value in SNAPSHOT.items():
    raw = base64.b64decode(value)
    if hashlib.sha256(raw).hexdigest() != EXPECTED[relative]:
        raise ValueError("Frozen snapshot differs: " + relative)
    rel = Path(relative)
    if rel.is_absolute() or ".." in rel.parts:
        raise ValueError("Invalid snapshot path")
    target = WORK_ROOT / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(raw)
sys.path.insert(0, str(WORK_ROOT / "code"))
print("Snapshot restored:", len(SNAPSHOT), "files.")


In [ ]:
from sena_experiment import SenaExperiment, validate_reviewed_data
checked = validate_reviewed_data(WORK_ROOT)
print({k: checked[k] for k in ["train_narratives", "validation_narratives", "train_class_counts", "validation_class_counts"]})
print("Baseline tersimpan:", checked["baseline"]["metrics_reviewed"]["accuracy"], checked["baseline"]["metrics_reviewed"]["macro_f1_all_five_classes"])
print("Tidak ada inferensi baseline ulang.")


## 3. Drive dan sumber model

ZIP SILAR_SENA_v1.zip sudah berada di folder yang sama seperti pekerjaan sebelumnya. Hash sumber diperiksa sebelum training.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_FOLDER = Path("/content/drive/MyDrive/SILAR_SLR09")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
experiment = SenaExperiment(WORK_ROOT, DRIVE_FOLDER)
print("Run baru:", experiment.run_id)


## 4. Pemuatan dan pemeriksaan alignment input

Memastikan GPU, hash bobot, urutan lima label, input tidak terpotong dan pembagian data.


In [ ]:
prepared = experiment.prepare()
print(json.dumps(prepared, ensure_ascii=False, indent=2))


## 5. Fine-tune baru

Maksimal 10 epoch dan patience 2. Checkpoint dipilih dari macro F1 lima kelas validation. Test tidak dipakai.


In [ ]:
training = experiment.fit()
print(json.dumps(training, ensure_ascii=False, indent=2))


## 6. Evaluasi kandidat baru

Menghasilkan precision/recall/F1 per kelas, confusion matrix, kesalahan NON_EVENT, kurva dan analisis cutoff. Hasil baseline dipakai dari perhitungan yang sudah selesai.


In [ ]:
evaluation = experiment.evaluate()
print(json.dumps(evaluation, ensure_ascii=False, indent=2))


## 7. Ekspor dan pemeriksaan reload

Model diuji reload pada tiga contoh dan disimpan ke Drive. Hasil kecil tersimpan terpisah dari bobot.


In [ ]:
saved = experiment.export()
print(json.dumps(saved, ensure_ascii=False, indent=2))


## Hasil yang dikirim kembali

Kirim **SLR09_SENA_Hasil_FineTune_<run_id>.zip**. Model besar tetap di Drive. Jika gagal, ZIP hasil sementara menyimpan stage/status; periksa sebelum memulai training baru. Tidak perlu menjalankan notebook NARA kembali.


## Lampiran kode lengkap: sena_experiment.py

```python
"""New SENA pilot fine-tune; reuse the baseline predictions already evaluated.

This file prepares a NEW experiment. It is not evidence that training ran.
110 reviewed train narratives and 620 reviewed validation narratives retain
their existing roles. Three deferred references and all official test IDs are
excluded. Validation is used for model selection, not described as blind test.
"""
from __future__ import annotations

import hashlib
import json
import os
import platform
import shutil
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

from evaluate_sena_reviewed import LABELS, cutoff_analysis, score


CONFIG = {
    "seed": 42, "max_length": 256, "learning_rate": 2e-5,
    "train_batch_size": 16, "eval_batch_size": 32,
    "max_epochs": 10, "weight_decay": 0.01,
    "early_stopping_patience": 2, "metric_for_best_model": "macro_f1",
    "transformers_version": "4.57.1",
}
BASELINE_ARCHIVE_SHA256 = "e4af9d1e09023bcfb29670f2ff93d9aa477e8af6990d1ad21192d50486de57f6"
BASELINE_WEIGHTS_SHA256 = "ae7ceb9959b7f0f3c55a6babd2cf73759a1d9458daadb654ca2fb68ca25cc4dc"


def file_hash(path: Path) -> str:
    hasher = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


def save_json(path: Path, value: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def read_rows(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def validate_reviewed_data(root: Path) -> dict:
    """Standard-library validation; may be run without loading a model."""
    manifest = json.loads((root / "data/input_manifest.json").read_text())
    for relative, expected in manifest["sha256"].items():
        if file_hash(root / relative) != expected:
            raise ValueError("Frozen input differs: " + relative)
    train = read_rows(root / "data/sena_train_reviewed.jsonl")
    val = read_rows(root / "data/sena_validation_reviewed.jsonl")
    if len(train) != 110 or len(val) != 620:
        raise ValueError("Expected the existing 110 train and 620 validation references")
    train_ids = [r["id"] for r in train]
    val_ids = [r["id"] for r in val]
    if len(set(train_ids)) != len(train_ids) or len(set(val_ids)) != len(val_ids):
        raise ValueError("Duplicate narrative ID")
    if set(train_ids) & set(val_ids):
        raise ValueError("Train/validation overlap")
    for r in train + val:
        if r["final"] not in LABELS or r["status"] not in {"REVIEW_TERCATAT", "FINAL_TERWARIS"}:
            raise ValueError("Reference is not final: " + r["id"])
        if int(r["id"].split("-")[-1]) <= 250:
            raise ValueError("Official held-out source ID must not enter fine-tune: " + r["id"])
    pending_ids = {"BAN-01028", "KEB-01113", "BAN-01529"}
    if pending_ids & (set(train_ids) | set(val_ids)):
        raise ValueError("Deferred reference entered fine-tune")
    if {r["text"].casefold().strip() for r in train} & {r["text"].casefold().strip() for r in val}:
        raise ValueError("Exact text duplicate across train/validation")
    baseline = json.loads((root / "data/baseline_reviewed_evaluation.json").read_text())
    if baseline["records_scored"] != 620 or baseline["production_threshold_selected"]:
        raise ValueError("Unexpected baseline scope or selected threshold")
    return {"train": train, "validation": val, "baseline": baseline,
            "train_narratives": len(train), "validation_narratives": len(val),
            "train_class_counts": {label: sum(r["final"] == label for r in train) for label in LABELS},
            "validation_class_counts": {label: sum(r["final"] == label for r in val) for label in LABELS},
            "test_used_for_training_or_selection": False,
            "historical_sena_train_membership_verified": False}


class SenaExperiment:
    def __init__(self, root: Path, drive_folder: Path, config: dict | None = None):
        self.root = Path(root)
        self.drive = Path(drive_folder)
        self.config = dict(CONFIG if config is None else config)
        self.run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        self.work = self.root / "runs" / self.run_id
        self.results = self.work / "results"
        self.results.mkdir(parents=True, exist_ok=True)
        self.state = {"schema": "silar-sena-finetune-run-0.1", "run_id": self.run_id,
            "status": "INITIALIZED", "completed_stages": [], "config": self.config,
            "started_utc": datetime.now(timezone.utc).isoformat(),
            "training_attempted": False, "training_executed": False, "training_completed": False,
            "new_neural_inference_started": False, "new_neural_inference_executed": False, "baseline_neural_inference_repeated": False,
            "model_exported": False, "production_model_selected": False,
            "production_threshold_selected": False,
            "validation_is_independent_blind_test": False}
        save_json(self.results / "run_state.json", self.state)

    def _save_results_zip(self) -> Path:
        self.drive.mkdir(parents=True, exist_ok=True)
        target = self.drive / f"SLR09_SENA_Hasil_FineTune_{self.run_id}.zip"
        with ZipFile(target, "w", ZIP_DEFLATED) as archive:
            for path in sorted(self.results.rglob("*")):
                if path.is_file():
                    archive.write(path, path.relative_to(self.results).as_posix())
        return target

    def _stage(self, name: str, function):
        self.state.update(status="RUNNING", current_stage=name)
        save_json(self.results / "run_state.json", self.state)
        try:
            result = function()
            self.state["completed_stages"].append(name)
            self.state["status"] = "SUCCEEDED" if name == "EXPORT" else "STAGE_SUCCEEDED"
            save_json(self.results / "run_state.json", self.state)
            self._save_results_zip()
            return result
        except Exception as error:
            if name == "FINE_TUNE" and hasattr(self, "trainer"):
                self.state["optimizer_steps"] = int(self.trainer.state.global_step)
                self.state["training_executed"] = self.trainer.state.global_step > 0
            self.state.update(status="FAILED", error_type=type(error).__name__, error=str(error))
            (self.results / "error_traceback.txt").write_text(traceback.format_exc())
            save_json(self.results / "run_state.json", self.state)
            try:
                print("ZIP hasil sementara:", self._save_results_zip())
            except Exception as saving_error:
                print("Hasil sementara lokal:", self.results, type(saving_error).__name__)
            raise

    def prepare(self) -> dict:
        return self._stage("PREPARE", self._prepare)

    def _prepare(self) -> dict:
        import torch
        import transformers
        from transformers import AutoModelForSequenceClassification, AutoTokenizer, set_seed
        if transformers.__version__ != self.config["transformers_version"]:
            raise RuntimeError("Use the pinned Transformers version and restart a previously used session")
        if not torch.cuda.is_available():
            raise RuntimeError("Pilih Runtime > Change runtime type > GPU di Colab; tahap ini adalah training baru")
        set_seed(self.config["seed"])
        self.data = validate_reviewed_data(self.root)
        self.train_rows = self.data["train"]
        self.val_rows = self.data["validation"]
        self.baseline = self.data["baseline"]["metrics_reviewed"]
        self.state.update(train_narratives=len(self.train_rows), validation_narratives=len(self.val_rows))
        source = self.drive / "SILAR_SENA_v1.zip"
        if not source.is_file() or file_hash(source) != BASELINE_ARCHIVE_SHA256:
            raise ValueError("ZIP SENA v1 pada Drive tidak ditemukan atau hash berbeda; jangan memakai model NARA")
        model_dir = self.work / "source_model"
        with ZipFile(source) as archive:
            if archive.testzip() is not None:
                raise ValueError("ZIP SENA v1 rusak")
            for info in archive.infolist():
                relative = Path(info.filename)
                if relative.is_absolute() or ".." in relative.parts:
                    raise ValueError("Invalid archive path")
                if info.is_dir():
                    continue
                destination = model_dir / relative
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(archive.read(info.filename))
        candidates = [p.parent for p in model_dir.rglob("config.json") if (p.parent / "model.safetensors").is_file()]
        if len(candidates) != 1:
            raise ValueError("Expected one exported SENA model root")
        model_path = candidates[0]
        if file_hash(model_path / "model.safetensors") != BASELINE_WEIGHTS_SHA256:
            raise ValueError("SENA baseline weight identity differs")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True, use_fast=True)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)
        if [self.model.config.id2label[i] for i in range(len(LABELS))] != LABELS:
            raise ValueError("SENA five-label mapping differs")
        self.label2id = {label: i for i, label in enumerate(LABELS)}
        max_length = self.config["max_length"]
        lengths = [len(self.tokenizer(r["text"], truncation=False)["input_ids"]) for r in self.train_rows + self.val_rows]
        if max(lengths) > max_length:
            raise ValueError("Input exceeds the pinned maximum; review truncation before training")
        environment = {"python": platform.python_version(), "torch": torch.__version__,
                       "transformers": transformers.__version__, "gpu": torch.cuda.get_device_name(0),
                       "cuda": torch.version.cuda, "source_archive_sha256": BASELINE_ARCHIVE_SHA256,
                       "source_weights_sha256": BASELINE_WEIGHTS_SHA256}
        save_json(self.results / "environment.json", environment)
        summary = {k: v for k, v in self.data.items() if k not in {"train", "validation", "baseline"}}
        summary.update(max_wordpieces=max(lengths), truncated_narratives=0)
        save_json(self.results / "dataset_manifest.json", summary)
        save_json(self.results / "baseline_reviewed_metrics.json", self.baseline)
        save_json(self.results / "input_manifest.json", json.loads((self.root / "data/input_manifest.json").read_text()))
        return summary

    def fit(self) -> dict:
        return self._stage("FINE_TUNE", self._fit)

    def _fit(self) -> dict:
        import numpy as np
        from torch.utils.data import Dataset
        from transformers import DataCollatorWithPadding, EarlyStoppingCallback, Trainer, TrainingArguments

        class TextDataset(Dataset):
            def __init__(self, rows, tokenizer, label2id):
                self.values = []
                for row in rows:
                    encoded = tokenizer(row["text"], truncation=False)
                    encoded["labels"] = label2id[row["final"]]
                    self.values.append(encoded)
            def __len__(self):
                return len(self.values)
            def __getitem__(self, item):
                return self.values[item]

        def compute_metrics(evaluation):
            logits, ids = evaluation
            if isinstance(logits, tuple):
                logits = logits[0]
            result = score([LABELS[int(i)] for i in ids], [LABELS[int(i)] for i in np.argmax(logits, axis=1)])
            return {"accuracy": result["accuracy"], "macro_f1": result["macro_f1_all_five_classes"],
                    "weighted_f1": result["weighted_f1"]}

        args = TrainingArguments(
            output_dir=str(self.work / "checkpoints"), learning_rate=self.config["learning_rate"],
            per_device_train_batch_size=self.config["train_batch_size"],
            per_device_eval_batch_size=self.config["eval_batch_size"],
            num_train_epochs=self.config["max_epochs"], weight_decay=self.config["weight_decay"],
            eval_strategy="epoch", save_strategy="epoch", logging_strategy="epoch",
            load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
            save_total_limit=2, seed=self.config["seed"], data_seed=self.config["seed"],
            report_to="none", fp16=False, dataloader_num_workers=0,
        )
        self.val_dataset = TextDataset(self.val_rows, self.tokenizer, self.label2id)
        self.trainer = Trainer(model=self.model, args=args,
            train_dataset=TextDataset(self.train_rows, self.tokenizer, self.label2id),
            eval_dataset=self.val_dataset, processing_class=self.tokenizer,
            data_collator=DataCollatorWithPadding(self.tokenizer), compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=self.config["early_stopping_patience"])])
        self.state["training_attempted"] = True
        save_json(self.results / "run_state.json", self.state)
        output = self.trainer.train()
        self.model = self.trainer.model
        self.state.update(training_executed=self.trainer.state.global_step > 0,
            training_completed=True, optimizer_steps=self.trainer.state.global_step,
            best_checkpoint=self.trainer.state.best_model_checkpoint,
            best_validation_macro_f1=self.trainer.state.best_metric,
            epochs_completed=self.trainer.state.epoch)
        save_json(self.results / "training_result.json", output.metrics)
        save_json(self.results / "training_log_history.json", self.trainer.state.log_history)
        self._plot_training()
        return {k: self.state[k] for k in ["optimizer_steps", "best_checkpoint", "best_validation_macro_f1", "epochs_completed"]}

    def _plot_training(self):
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        history = self.trainer.state.log_history
        train = [r for r in history if "loss" in r]
        val = [r for r in history if "eval_loss" in r]
        fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
        axes[0].plot([r["epoch"] for r in train], [r["loss"] for r in train], label="Train")
        axes[0].plot([r["epoch"] for r in val], [r["eval_loss"] for r in val], label="Validation")
        axes[0].set(xlabel="Epoch", ylabel="Loss", title="SENA pilot: loss"); axes[0].legend()
        axes[1].plot([r["epoch"] for r in val], [r["eval_macro_f1"] for r in val])
        axes[1].set(xlabel="Epoch", ylabel="Macro F1 (5 classes)", title="Validation used for checkpoint selection", ylim=(0, 1))
        fig.savefig(self.results / "training_curves.png", dpi=160); plt.close(fig)

    def evaluate(self) -> dict:
        return self._stage("EVALUATE", self._evaluate)

    def _evaluate(self) -> dict:
        import numpy as np
        import torch
        from evaluate_sena_reviewed import write_csv
        self.state["new_neural_inference_started"] = True
        save_json(self.results / "run_state.json", self.state)
        output = self.trainer.predict(self.val_dataset)
        logits = output.predictions[0] if isinstance(output.predictions, tuple) else output.predictions
        probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
        preds = np.argmax(probs, axis=1)
        rows = [{"id": r["id"], "text": r["text"], "gold": r["final"],
                 "prediction": LABELS[int(p)], "confidence": float(prob[int(p)]),
                 "probabilities": {label: float(prob[j]) for j, label in enumerate(LABELS)}}
                for r, p, prob in zip(self.val_rows, preds, probs)]
        metrics = score([r["gold"] for r in rows], [r["prediction"] for r in rows])
        save_json(self.results / "finetuned_validation_metrics.json", metrics)
        (self.results / "finetuned_validation_predictions.jsonl").write_text(
            "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")
        save_json(self.results / "cutoff_analysis.json", {"rows": cutoff_analysis(rows),
            "production_threshold_selected": False, "independent_test": False})
        write_csv(self.results / "per_class.csv", [{"label": label, **v} for label, v in metrics["per_class"].items()])
        errors = [r for r in rows if r["gold"] != r["prediction"]]
        save_json(self.results / "classification_errors.json", errors)
        comparison = {"baseline_reused_without_neural_rerun": self.baseline, "finetuned_validation": metrics,
            "validation_used_for_checkpoint_selection": True, "independent_blind_test": False,
            "historical_training_split_identity_verified": False, "official_test_evaluated": False,
            "new_model_production_ready": False}
        save_json(self.results / "comparison.json", comparison)
        self._plot_matrix(metrics)
        self.state["new_neural_inference_executed"] = True
        return {"accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1_all_five_classes"],
                "false_non_event": metrics["false_non_event_count"], "evaluated_narratives": len(rows)}

    def _plot_matrix(self, metrics):
        import csv
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        matrix = metrics["confusion_matrix_rows_gold_columns_prediction"]
        with (self.results / "confusion_matrix.csv").open("w", newline="", encoding="utf-8") as stream:
            writer = csv.writer(stream); writer.writerow(["gold/prediction", *LABELS])
            for label, row in zip(LABELS, matrix): writer.writerow([label, *row])
        fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
        plot = ax.imshow(matrix, cmap="Blues"); fig.colorbar(plot, ax=ax, shrink=.8)
        ax.set(xticks=range(5), yticks=range(5), xticklabels=LABELS, yticklabels=LABELS,
               xlabel="Prediction", ylabel="Reviewed reference", title="SENA: 620 validation narratives")
        plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
        for i, row in enumerate(matrix):
            for j, number in enumerate(row): ax.text(j, i, str(number), ha="center", va="center")
        fig.savefig(self.results / "confusion_matrix.png", dpi=160); plt.close(fig)

    def export(self) -> dict:
        return self._stage("EXPORT", self._export)

    def _export(self) -> dict:
        import torch
        from transformers import AutoModelForSequenceClassification
        export = self.work / "export_model"
        self.trainer.save_model(export); self.tokenizer.save_pretrained(export)
        metadata = {"project": "SILAR", "model": "SENA reviewed pilot", "run_id": self.run_id,
            "classes": LABELS, "config": self.config,
            "source_weights_sha256": BASELINE_WEIGHTS_SHA256,
            "train_ids": [r["id"] for r in self.train_rows],
            "validation_ids": [r["id"] for r in self.val_rows],
            "official_test_used_for_training_or_selection": False,
            "historical_base_training_membership_verified": False,
            "production_model_selected": False, "production_threshold_selected": False}
        save_json(export / "experiment_metadata.json", metadata)
        samples = self.tokenizer([r["text"] for r in self.val_rows[:3]], padding=True, return_tensors="pt")
        device = next(self.model.parameters()).device
        samples = {key: value.to(device) for key, value in samples.items()}
        self.model.eval()
        with torch.no_grad(): before = self.model(**samples).logits.detach().cpu()
        reloaded = AutoModelForSequenceClassification.from_pretrained(export, local_files_only=True).to(device).eval()
        with torch.no_grad(): after = reloaded(**samples).logits.detach().cpu()
        if not torch.allclose(before, after, atol=1e-5, rtol=1e-5):
            raise ValueError("Model reload logits differ on the three verification samples")
        del reloaded
        target = self.drive / f"SILAR_SENA_reviewed_pilot_{self.run_id}.zip"
        with ZipFile(target, "w", ZIP_DEFLATED) as archive:
            for p in sorted(export.rglob("*")):
                if p.is_file(): archive.write(p, p.relative_to(export).as_posix())
        manifest = {"archive_name": target.name, "archive_sha256": file_hash(target),
            "files_sha256": {p.relative_to(export).as_posix(): file_hash(p) for p in sorted(export.rglob("*")) if p.is_file()},
            "reload_sample_count": 3, "reload_logits_match": True,
            "production_model_selected": False, "production_threshold_selected": False}
        save_json(self.results / "model_export_manifest.json", manifest)
        self.state.update(model_exported=True, model_archive_path=str(target),
                          finished_utc=datetime.now(timezone.utc).isoformat())
        return {"model_zip": str(target),
                "results_zip": str(self.drive / f"SLR09_SENA_Hasil_FineTune_{self.run_id}.zip"),
                "reload_logits_match": True, "production_model_selected": False}

```


## Lampiran kode lengkap: evaluate_sena_reviewed.py

```python
"""Evaluate saved SENA predictions against the completed B1 review.

No model, training, new neural inference, probability fitting, or production
threshold selection is performed. This analysis adds the 620 final human
references; the previous analysis used 71 reviewed and 552 original references.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import math
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from zipfile import ZipFile


LABELS = ["FLOOD", "EARTHQUAKE", "FIRE", "ACCIDENT", "NON_EVENT"]
THRESHOLDS = [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0]
SENA_WEIGHTS_SHA256 = "ae7ceb9959b7f0f3c55a6babd2cf73759a1d9458daadb654ca2fb68ca25cc4dc"


def digest(raw: bytes) -> str:
    return hashlib.sha256(raw).hexdigest()


def jsonl(raw: bytes) -> list[dict]:
    return [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]


def unique_index(rows: list[dict], label: str) -> dict[str, dict]:
    result = {row["id"]: row for row in rows}
    if len(result) != len(rows):
        raise ValueError(f"Duplicate IDs: {label}")
    return result


def score(gold: list[str], pred: list[str]) -> dict:
    if len(gold) != len(pred) or not gold:
        raise ValueError("Gold/prediction coverage differs or is empty")
    index = {label: i for i, label in enumerate(LABELS)}
    matrix = [[0 for _ in LABELS] for _ in LABELS]
    for g, p in zip(gold, pred):
        if g not in index or p not in index:
            raise ValueError("Unknown SENA category")
        matrix[index[g]][index[p]] += 1
    report = {}
    for i, label in enumerate(LABELS):
        tp = matrix[i][i]
        support = sum(matrix[i])
        predicted = sum(row[i] for row in matrix)
        precision = tp / predicted if predicted else 0.0
        recall = tp / support if support else 0.0
        f1 = 2 * tp / (support + predicted) if support + predicted else 0.0
        report[label] = dict(support=support, predicted=predicted, precision=precision, recall=recall, f1=f1)
    correct = sum(matrix[i][i] for i in range(len(LABELS)))
    event_count = sum(g != "NON_EVENT" for g in gold)
    false_non_event = sum(g != "NON_EVENT" and p == "NON_EVENT" for g, p in zip(gold, pred))
    false_event = sum(g == "NON_EVENT" and p != "NON_EVENT" for g, p in zip(gold, pred))
    wrong_event_type = sum(g != p and g != "NON_EVENT" and p != "NON_EVENT" for g, p in zip(gold, pred))
    return {
        "records": len(gold), "correct": correct, "incorrect": len(gold) - correct,
        "accuracy": correct / len(gold),
        "macro_f1_all_five_classes": sum(v["f1"] for v in report.values()) / len(LABELS),
        "weighted_f1": sum(v["f1"] * v["support"] for v in report.values()) / len(gold),
        "per_class": report,
        "labels": LABELS, "confusion_matrix_rows_gold_columns_prediction": matrix,
        "gold_event_count": event_count, "false_non_event_count": false_non_event,
        "false_non_event_rate_among_gold_events": false_non_event / event_count if event_count else None,
        "false_event_count": false_event, "wrong_event_type_count": wrong_event_type,
    }


def cutoff_analysis(rows: list[dict]) -> list[dict]:
    results = []
    for threshold in THRESHOLDS:
        high = [r for r in rows if r["confidence"] >= threshold]
        low = [r for r in rows if r["confidence"] < threshold]
        correct_high = sum(r["gold"] == r["prediction"] for r in high)
        errors_high = len(high) - correct_high
        errors_low = sum(r["gold"] != r["prediction"] for r in low)
        all_errors = errors_high + errors_low
        results.append({
            "threshold": threshold,
            "below_threshold_uncertain": len(low), "at_or_above_threshold": len(high),
            "coverage_at_or_above": len(high) / len(rows),
            "correct_at_or_above": correct_high,
            "accuracy_at_or_above": correct_high / len(high) if high else None,
            "errors_at_or_above": errors_high,
            "errors_below_threshold": errors_low,
            "fraction_of_errors_flagged": errors_low / all_errors if all_errors else None,
            "false_non_event_at_or_above": sum(r["gold"] != "NON_EVENT" and r["prediction"] == "NON_EVENT" for r in high),
            "non_event_predictions_blocked_total": sum(r["prediction"] == "NON_EVENT" for r in rows),
        })
    return results


def write_json(path: Path, value: object) -> None:
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")


def write_csv(path: Path, rows: list[dict], columns: list[str] | None = None) -> None:
    columns = columns or (list(rows[0]) if rows else [])
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)


def evaluate(runtime_zip: Path, review_jsonl: Path, review_checkpoint: Path,
             review_manifest: Path, reviewed_workbook: Path, output: Path) -> dict:
    raw_review = review_jsonl.read_bytes()
    review = jsonl(raw_review)
    cp = json.loads(review_checkpoint.read_bytes())
    manifest_bytes = review_manifest.read_bytes()
    workbook_bytes = reviewed_workbook.read_bytes()
    checks = []

    def require(condition: bool, name: str) -> None:
        checks.append({"check": name, "passed": bool(condition)})
        if not condition:
            raise ValueError(name)

    require(digest(manifest_bytes) == cp["manifest_sha256"], "frozen review manifest SHA-256")
    require(digest(workbook_bytes) == cp["workbook_sha256"], "human-reviewed workbook SHA-256")
    require(not cp["validation_errors"], "review checkpoint contains no validation errors")
    approved = [r for r in cp["approved"] if r.get("cohort") == "sena_validation"]
    require(unique_index(review, "review JSONL") == unique_index(approved, "checkpoint approved"),
            "620 exported references exactly match approved checkpoint")
    require(len(review) == 620, "620 final SENA references")
    require(all(r["status"] in {"REVIEW_TERCATAT", "FINAL_TERWARIS"} for r in review),
            "only final human review statuses scored")
    require(all(r["final"] in LABELS for r in review), "all reviewed categories recognized")
    pending = [r for r in cp["pending"] if r["sheet"] == "SENA_Validation"]
    require({r["id"] for r in pending} == {"BAN-01028", "KEB-01113", "BAN-01529"},
            "three deferred SENA cases unchanged and excluded")

    with ZipFile(runtime_zip) as archive:
        require(archive.testzip() is None, "runtime archive CRC")
        candidate_bytes = archive.read("results/candidate_split/candidate_validation.jsonl")
        prediction_bytes = archive.read("results/sena_validation/predictions.jsonl")
        split = json.loads(archive.read("results/candidate_split/split_manifest.json"))
        run = json.loads(archive.read("results/sena_validation/run_summary.json"))
        require(digest(candidate_bytes) == split["files_sha256"]["candidate_validation.jsonl"],
                "candidate narrative snapshot SHA-256")
        require(digest(prediction_bytes) == run["predictions_sha256"], "saved predictions SHA-256")
        require(run["status"] == "SUCCEEDED" and run["records_completed"] == 623,
                "successful 623-record original SENA inference")
        require(run["models"]["sena"]["files"]["model.safetensors"] == SENA_WEIGHTS_SHA256,
                "fixed SENA v1 weight identity in original run")
    candidate = unique_index(jsonl(candidate_bytes), "candidate narratives")
    predictions = unique_index(jsonl(prediction_bytes), "saved predictions")
    require(set(candidate) == set(predictions) and len(candidate) == 623, "all original candidate IDs covered")
    require(set(candidate) == {r["id"] for r in review} | {r["id"] for r in pending},
            "620 scored plus three omitted cover exactly the original 623")
    rows = []
    for ref in review:
        rid = ref["id"]
        case = candidate[rid]
        prediction = predictions[rid]
        require(ref["text"] == case["text"] and digest(ref["text"].encode("utf-8")) == prediction["input_sha256"],
                f"{rid}: exact narrative and inference input identity")
        classifier = prediction["event_classifier"]
        probs = classifier["probabilities"]
        require(set(probs) == set(LABELS) and all(isinstance(v, (int, float)) and not isinstance(v, bool)
                and math.isfinite(v) and 0 <= v <= 1 for v in probs.values()), f"{rid}: five finite probabilities")
        require(abs(sum(probs.values()) - 1) < 2e-5 and classifier["label"] == max(probs, key=probs.get)
                and abs(classifier["confidence"] - probs[classifier["label"]]) < 1e-6,
                f"{rid}: confidence and argmax consistent")
        require(not prediction["sena_token_coverage"]["truncated"], f"{rid}: input not truncated")
        rows.append({"id": rid, "text": ref["text"], "gold": ref["final"],
                     "original_gold": case["gold_label"], "prediction": classifier["label"],
                     "confidence": classifier["confidence"], "review_status": ref["status"],
                     "review_note": ref["note"], "input_sha256": prediction["input_sha256"]})
    gold = [r["gold"] for r in rows]
    pred = [r["prediction"] for r in rows]
    reviewed_metrics = score(gold, pred)
    original_metrics = score([r["original_gold"] for r in rows], pred)
    cutoffs = cutoff_analysis(rows)
    errors = []
    for r in rows:
        if r["gold"] == r["prediction"]:
            continue
        kind = "FALSE_NON_EVENT" if r["prediction"] == "NON_EVENT" else (
            "FALSE_EVENT" if r["gold"] == "NON_EVENT" else "WRONG_EVENT_TYPE")
        errors.append({**r, "error_kind": kind})
    result = {
        "schema": "slr09-sena-reviewed-evaluation-0.1",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "scope": "REVIEWED_620_OF_CANDIDATE_623_NOT_INDEPENDENT_TEST",
        "records_original": 623, "records_scored": len(rows), "records_omitted": len(pending),
        "omitted_cases": pending, "metrics_reviewed": reviewed_metrics,
        "metrics_original_gold_same_620": original_metrics,
        "reference_labels_changed": sum(r["gold"] != r["original_gold"] for r in rows),
        "cutoff_analysis": cutoffs,
        "new_training_executed": False, "new_neural_inference_executed": False,
        "probability_calibration_fitted": False, "production_threshold_selected": False,
        "production_confidence_threshold": None,
        "historical_training_split_identity_verified": False,
        "reviewed_reference_is_new_blind_test": False,
        "non_event_always_blocks_submission": True,
        "human_verification_required_at_all_confidences": True,
        "source_sha256": {"runtime_zip": digest(runtime_zip.read_bytes()),
            "predictions": digest(prediction_bytes), "review_jsonl": digest(raw_review),
            "review_checkpoint": digest(review_checkpoint.read_bytes()),
            "review_manifest": digest(manifest_bytes), "workbook": digest(workbook_bytes)},
        "original_run_summary": run, "original_split_manifest": split,
        "checks": checks,
        "limitations": [
            "Kandidat validation direkonstruksi; identitas terhadap training historis SENA belum terbukti.",
            "Review dilakukan setelah analisis prediksi; angka ini merupakan analisis retrospektif.",
            "Tiga kasus ambigu tetap ditunda; 620 bukan seluruh 623 kasus.",
            "Cutoff hanya menandai confidence rendah; tidak memperbaiki label atau membatalkan NON_EVENT.",
            "Tidak memilih target mutu, model produksi, threshold produksi, atau melatih temperature scaling.",
        ],
    }
    output.mkdir(parents=True, exist_ok=True)
    write_json(output / "evaluation.json", result)
    write_json(output / "omitted_cases.json", pending)
    write_json(output / "scored_rows.json", rows)
    write_csv(output / "classification_errors.csv", errors, ["id", "text", "gold", "original_gold",
              "prediction", "confidence", "review_status", "review_note", "input_sha256", "error_kind"])
    write_csv(output / "threshold_candidates.csv", cutoffs)
    write_csv(output / "per_class.csv", [{"label": k, **v} for k, v in reviewed_metrics["per_class"].items()])
    with (output / "confusion_matrix.csv").open("w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(["gold/prediction", *LABELS])
        for label, row in zip(LABELS, reviewed_metrics["confusion_matrix_rows_gold_columns_prediction"]):
            writer.writerow([label, *row])
    return result


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--runtime-zip", type=Path, required=True)
    parser.add_argument("--review-jsonl", type=Path, required=True)
    parser.add_argument("--review-checkpoint", type=Path, required=True)
    parser.add_argument("--review-manifest", type=Path, required=True)
    parser.add_argument("--reviewed-workbook", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    args = parser.parse_args()
    result = evaluate(args.runtime_zip, args.review_jsonl, args.review_checkpoint,
                      args.review_manifest, args.reviewed_workbook, args.output_dir)
    print(json.dumps({"scope": result["scope"], "scored": result["records_scored"],
        "accuracy": result["metrics_reviewed"]["accuracy"],
        "macro_f1": result["metrics_reviewed"]["macro_f1_all_five_classes"],
        "false_non_event": result["metrics_reviewed"]["false_non_event_count"],
        "checks_passed": len(result["checks"]), "new_neural_inference": False}, indent=2))


if __name__ == "__main__":
    main()

```
